# PyTorch-এর মৌলিক বিষয়

Tensors, autograd, nn.Module, loss/optimizer এবং মানক training loop —
`02-Neural-Networks-Basics/example.py`-র XOR network-টি নতুন করে তৈরি
করা, তবে backward pass হাতে করার বদলে PyTorch-এর autograd দিয়ে হিসাব
করা।

প্রয়োজন: torch (`pip install torch`)

চালানোর নিয়ম:
    cell-গুলো উপরে থেকে নিচে চালাও (Shift + Enter)।


In [ ]:
"""
PyTorch Fundamentals

Tensors, autograd, nn.Module, loss/optimizer এবং মানক training loop —
`02-Neural-Networks-Basics/example.py`-র ঠিক একই XOR network-টি নবনির্মাণ,
তবে backward pass হাতে করার বদলে PyTorch-এর autograd দিয়ে হিসাব করা।

প্রয়োজন: torch (pip install torch)

চালানোর নিয়ম:
    এই notebook-এর cell-গুলো উপরে থেকে নিচে চালাও (Shift + Enter)।
"""

import torch
import torch.nn as nn


## 1. Tensor basics

`torch.Tensor` হলো NumPy-র `ndarray` প্লাস দুটি সুপারপাওয়ার: GPU-তে
থাকা এবং automatic differentiation-এর জন্য operations ট্র্যাক করা।


In [ ]:
# ---------------------------------------------------------------------------
# 1. Tensor basics
# ---------------------------------------------------------------------------

def tensor_basics_demo():
    print("=" * 70)
    print("1. TENSOR BASICS")
    print("=" * 70)

    x = torch.tensor([1.0, 2.0, 3.0])
    print(f"x = {x}, shape={tuple(x.shape)}, dtype={x.dtype}, device={x.device}")

    W = torch.tensor([[1.0, 0.0, -1.0], [0.5, 2.0, 1.0]])   # shape (2, 3)
    y = W @ x                                                # matrix-vector product
    print(f"W shape={tuple(W.shape)} @ x shape={tuple(x.shape)} -> y = {y}")

    # এই কোর্সে সর্বত্র ব্যবহৃত device-agnostic pattern
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Selected device: {device}")


tensor_basics_demo()


## 2. Autograd: automatic differentiation বনাম Lesson 1-এর ম্যানুয়াল হিসাব

`requires_grad=True` ও `.backward()`-এর মাধ্যমে PyTorch নিজেই
computational graph তৈরি করে এবং `.grad`-এ derivatives পূরণ করে।


In [ ]:
# ---------------------------------------------------------------------------
# 2. Autograd: automatic differentiation vs. the manual math from Lesson 1
# ---------------------------------------------------------------------------

def autograd_demo():
    print("\n" + "=" * 70)
    print("2. AUTOGRAD")
    print("=" * 70)

    # f(x) = x^2 -> f'(x) = 2x — math refresher-এ ব্যবহৃত একই ফাংশন।
    x = torch.tensor(3.0, requires_grad=True)
    y = x ** 2
    y.backward()
    print(f"x = {x.item()}, y = x^2 = {y.item()}")
    print(f"autograd x.grad = {x.grad.item()}  (analytic 2x = {2 * x.item()})")

    # Chain rule: y = sin(x^2) — math refresher-এর মতোই একই composed function।
    x2 = torch.tensor(1.5, requires_grad=True)
    y2 = torch.sin(x2 ** 2)
    y2.backward()
    analytic = torch.cos(x2.detach() ** 2) * (2 * x2.detach())
    print(f"\nd/dx sin(x^2) at x=1.5 -> autograd = {x2.grad.item():.6f}, "
          f"analytic = {analytic.item():.6f}")

    # no_grad / detach demo
    with torch.no_grad():
        y3 = x2 ** 2  # ট্র্যাক হয় না; graph তৈরি হয় না — inference-এ memory বাঁচায়
    print(f"\nUnder torch.no_grad(): y3.requires_grad = {y3.requires_grad}")


autograd_demo()


## 3. nn.Module: Lesson 2-এর মতোই XOR network, framework-পরিচালিত

`nn.Linear` weight matrix ও bias-কে একত্রিত করে ও register করে,
`nn.Sigmoid` activation হিসেবে।


In [ ]:
# ---------------------------------------------------------------------------
# 3. nn.Module: the same XOR network as Lesson 2, framework-managed
# ---------------------------------------------------------------------------

class TwoLayerNet(nn.Module):
    """02-Neural-Networks-Basics/example.py-র SimpleMLP-এর মতোই architecture:
    2 inputs -> 4 hidden (sigmoid) -> 1 output (sigmoid)।"""

    def __init__(self, n_in=2, n_hidden=4, n_out=1):
        super().__init__()
        self.fc1 = nn.Linear(n_in, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_out)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h = self.sigmoid(self.fc1(x))
        out = self.sigmoid(self.fc2(h))
        return out


## 4. মানক training loop (XOR, PyTorch সংস্করণ)

প্রতি epoch-এ পাঁচ ধাপ: zero_grad → forward pass → loss → backward →
step — Lesson 2-এর ম্যানুয়াল সংস্করণের কাঠামোই, autograd-স্বয়ংক্রিয়।


In [ ]:
# ---------------------------------------------------------------------------
# 4. The standard training loop (XOR, PyTorch version)
# ---------------------------------------------------------------------------

def training_loop_demo():
    print("\n" + "=" * 70)
    print("3. nn.Module + THE STANDARD TRAINING LOOP (XOR, PyTorch version)")
    print("=" * 70)

    torch.manual_seed(42)

    # এখানে shape সম্মেলন হলো (batch, features) — PyTorch-এর default —
    # যা Lesson 2-এর scratch থেকে লেখা NumPy সংস্করণে ব্যবহৃত
    # (features, batch) সম্মেলনের transpose।
    X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
    Y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

    model = TwoLayerNet()
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

    epochs = 5000
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()          # 1. পুরনো gradients মুছে ফেলা
        output = model(X)              # 2. forward pass
        loss = criterion(output, Y)    # 3. loss হিসাব করা
        loss.backward()                # 4. backward pass (autograd সব gradient হিসাব করে)
        optimizer.step()               # 5. gradient descent update

        if epoch % 1000 == 0 or epoch == 1:
            print(f"epoch {epoch:5d}  loss = {loss.item():.6f}")

    print(f"\nFinal loss: {loss.item():.6f}")
    with torch.no_grad():
        predictions = model(X)
    print("\nPredictions vs targets:")
    for i in range(X.shape[0]):
        pred = predictions[i, 0].item()
        target = Y[i, 0].item()
        print(f"  input={X[i].tolist()} -> predicted={pred:.4f}  "
              f"(rounded={round(pred)}, target={int(target)})")


training_loop_demo()


## সব একসাথে (main)

নিচের cell-এ পুরো স্ক্রিপ্টটি একবারে চালানো হয় — যেমনটি `python
example.py` চালালে হয়। (উপরের section cell-গুলোতে অংশগুলো আলাদাভাবে
দেখানো হয়েছে।)


In [ ]:
def main():
    tensor_basics_demo()
    autograd_demo()
    training_loop_demo()


main()
